# S3 J2 — Coordination multi-agents — Notebook formateur

Source générée depuis les fichiers Markdown du jour.

# Semaine 3 — Jour 2 — Coordination multi-agents

## Position dans le bootcamp

La semaine 3 introduit les systèmes multi-agents et MCP.  
Le jour 1 a défini les architectures multi-agents : manager-worker, handoff, router, reviewer et exécution parallèle.  
Le jour 2 se concentre sur la **coordination** : comment décider quel agent travaille, dans quel ordre, avec quel contexte, quelles règles de validation et quels mécanismes de résolution de conflit.

## Problème traité

Un système multi-agent n'est pas seulement un ensemble d'agents.  
Sans coordination explicite, il devient rapidement :

- non déterministe ;
- coûteux ;
- difficile à déboguer ;
- vulnérable aux boucles infinies ;
- incapable d'expliquer pourquoi une décision a été prise.

La coordination est la couche qui transforme plusieurs capacités isolées en workflow fiable.

## Objectif du jour

À la fin de cette journée, l'apprenant doit savoir construire un coordinateur multi-agent minimal capable de :

1. router une tâche vers les bons agents ;
2. construire un plan d'exécution ;
3. transmettre un contexte limité et utile ;
4. agréger plusieurs réponses ;
5. détecter les désaccords ;
6. décider si une synthèse est suffisante ou si une revue supplémentaire est nécessaire ;
7. produire une trace exploitable.

## Livrables produits

```text
book/week03/day02/
├── README.md
├── learning_objectives.md
├── chapter.md
├── exercises.md
├── interview.md
├── challenge.md
├── references.md
├── corriges/
│   ├── exercises_solution.md
│   ├── interview_solution.md
│   ├── challenge_solution.md
│   └── review.md
├── diagrams/
│   ├── coordination_control_flow.mmd
│   └── disagreement_resolution_sequence.mmd
├── assets/
│   ├── manifest.json
│   ├── coordination_plan_schema.json
│   └── example_coordination_tasks.json
└── labs/
    ├── README.md
    ├── multi_agent_coordinator.py
    └── test_multi_agent_coordinator.py

notebooks/week03/
├── S3_J2_coordination.ipynb
└── teacher/
    └── S3_J2_coordination_teacher.ipynb
```

## Fil rouge

Le lab implémente un coordinateur déterministe pour un assistant IA mono-projet.  
L'utilisateur demande une analyse. Le coordinateur identifie les domaines nécessaires, assigne des agents spécialistes, collecte les résultats, fait intervenir un reviewer si nécessaire, puis génère une décision structurée.

## Compétence AI Engineering

La compétence clé n'est pas de créer beaucoup d'agents.  
La compétence clé est de créer une orchestration explicable, limitée, testable et observable.

# Objectifs pédagogiques — Jour 2 — Coordination

## Objectifs principaux

À la fin de cette journée, l'apprenant doit pouvoir :

- expliquer pourquoi la coordination est une couche distincte de l'intelligence d'un agent ;
- différencier routing, handoff, délégation, vote, revue et synthèse ;
- concevoir une politique de coordination explicite ;
- limiter le contexte transmis à chaque agent ;
- représenter une exécution multi-agent sous forme de trace ;
- détecter un désaccord entre agents ;
- déclencher une revue ou une clarification ;
- tester un coordinateur sans appeler un LLM réel.

## Objectifs techniques

L'apprenant doit savoir implémenter :

- un registre d'agents spécialistes ;
- une fonction de sélection d'agents à partir d'une tâche ;
- un plan de coordination sérialisable ;
- une boucle d'exécution bornée ;
- une stratégie d'agrégation ;
- une détection simple de conflit ;
- une synthèse finale structurée ;
- des tests unitaires pour valider la stabilité du workflow.

## Objectifs d'architecture

L'apprenant doit être capable de justifier :

- pourquoi le coordinateur ne doit pas tout déléguer aveuglément ;
- pourquoi un agent ne doit recevoir que le contexte nécessaire ;
- pourquoi les handoffs doivent être observables ;
- pourquoi les décisions sensibles doivent passer par une étape de revue ;
- pourquoi la coordination doit être déterministe autant que possible.

## Critères de réussite

Une solution est considérée correcte si elle :

- assigne les tâches aux bons agents ;
- ne sélectionne pas d'agents inutiles ;
- conserve une trace claire ;
- produit un résultat stable pour une même entrée ;
- sait signaler un conflit ;
- déclenche une revue en cas de risque élevé ;
- reste exécutable sans dépendances externes.

# Chapitre — Coordination multi-agents

## 1. Pourquoi coordonner ?

Un système multi-agent peut paraître simple au départ :

```text
Utilisateur → Agent A → Agent B → Réponse
```

Mais dès que plusieurs agents disposent de compétences différentes, il faut répondre à des questions d'ingénierie :

- quel agent doit travailler ?
- quel agent doit décider ?
- quel contexte doit être partagé ?
- que faire si deux agents sont en désaccord ?
- comment éviter les boucles ?
- comment expliquer le résultat ?
- comment tester le workflow ?

La coordination est la réponse à ces questions.

Un coordinateur multi-agent est une couche applicative qui transforme une demande utilisateur en exécution contrôlée.

## 2. Coordination vs intelligence

Il faut éviter une erreur fréquente : confondre intelligence et orchestration.

Un agent spécialiste peut très bien produire une bonne réponse locale.  
Mais cela ne garantit pas que le système global soit fiable.

Exemple :

- l'agent `product` veut accélérer la livraison ;
- l'agent `security` refuse une action non validée ;
- l'agent `support` veut répondre vite ;
- l'agent `reviewer` veut réduire le risque.

Sans coordination, le système peut produire une réponse incohérente.

La coordination définit les règles de composition.

## 3. Les rôles habituels

Dans un système multi-agent, on trouve souvent les rôles suivants.

### 3.1 Router

Le router décide quels agents sont pertinents.

Exemple :

```text
Demande : "Analyse ce bug de paiement et prépare une réponse client."
Agents utiles :
- support
- engineering
- billing
```

Le router ne résout pas nécessairement la tâche.  
Il sélectionne les capacités.

### 3.2 Manager

Le manager construit le plan.

Il peut décider :

- de lancer des agents en parallèle ;
- d'imposer un ordre ;
- de déclencher une revue ;
- de limiter le nombre d'itérations.

### 3.3 Specialist

Le specialist produit un résultat local.

Exemples :

- agent juridique ;
- agent sécurité ;
- agent produit ;
- agent support ;
- agent data ;
- agent backend.

### 3.4 Reviewer

Le reviewer vérifie la cohérence, le risque et les contradictions.

Il ne doit pas être appelé systématiquement si le coût est un problème.  
Mais il devient essentiel pour les décisions sensibles.

### 3.5 Synthesizer

Le synthesizer transforme plusieurs résultats locaux en réponse finale.

Il ne doit pas masquer les désaccords critiques.  
Il doit signaler ce qui est certain, incertain ou bloquant.

## 4. Patterns de coordination

## 4.1 Manager-worker

Le manager garde le contrôle.  
Les agents spécialistes sont appelés comme outils.

Avantages :

- contrôle centralisé ;
- traçabilité claire ;
- facilité de test ;
- pas de transfert incontrôlé.

Inconvénients :

- manager potentiellement complexe ;
- goulot d'étranglement ;
- moins flexible pour les conversations longues.

## 4.2 Handoff

Un agent transfère le contrôle à un autre agent.

Avantages :

- spécialisation forte ;
- bon pour les conversations orientées domaine ;
- naturel pour triage → spécialiste.

Inconvénients :

- risque de perte de contexte ;
- risque de chaîne de transferts ;
- besoin de traces précises.

## 4.3 Debate / critique

Plusieurs agents produisent des avis, puis un arbitre tranche.

Avantages :

- utile pour les décisions ambiguës ;
- améliore la robustesse ;
- met en évidence les désaccords.

Inconvénients :

- plus coûteux ;
- peut renforcer des hallucinations si les agents partagent les mêmes faiblesses ;
- nécessite une politique de décision claire.

## 4.4 Pipeline

Chaque agent enrichit progressivement un artefact.

Exemple :

```text
Requirements → Architecture → Security Review → Implementation Plan → Final Answer
```

Avantages :

- structure claire ;
- adapté aux livrables longs ;
- facile à auditer.

Inconvénients :

- moins adapté aux demandes ouvertes ;
- blocage si une étape échoue.

## 5. Le contrat de coordination

Un coordinateur fiable doit manipuler des structures explicites.

Exemple de contrat :

```json
{
  "task_id": "task-001",
  "required_domains": ["support", "engineering"],
  "risk_level": "medium",
  "selected_agents": ["support_agent", "engineering_agent"],
  "requires_review": true,
  "max_rounds": 2
}
```

Ce contrat rend le système :

- testable ;
- sérialisable ;
- observable ;
- réutilisable.

## 6. Limiter le contexte

Un piège courant consiste à transmettre toute la conversation à tous les agents.

C'est souvent une mauvaise pratique :

- coût plus élevé ;
- fuite d'informations entre domaines ;
- perte de focus ;
- difficultés de debug ;
- risque de contamination du raisonnement.

Un coordinateur doit fournir à chaque agent un contexte minimal.

Exemple :

```text
Agent billing :
- problème : facture incorrecte
- client : entreprise
- montant : 240 €
- contrainte : répondre sans promettre de remboursement automatique
```

Il n'a pas besoin de recevoir tout l'historique technique.

## 7. Gestion des conflits

Deux agents peuvent produire des conclusions incompatibles.

Exemple :

```text
engineering: "Le bug est résolu."
support: "Le client rapporte encore l'erreur."
```

Le coordinateur doit détecter que la synthèse ne peut pas simplement dire :  
"Tout est résolu."

Stratégies possibles :

1. demander une clarification ;
2. déclencher une revue ;
3. produire une réponse avec incertitude explicite ;
4. relancer un agent avec observation complémentaire ;
5. refuser de conclure si le risque est trop élevé.

## 8. Traces et observabilité

Une exécution multi-agent doit produire une trace.

La trace doit contenir :

- l'entrée ;
- les agents sélectionnés ;
- l'ordre d'exécution ;
- les résultats locaux ;
- les conflits détectés ;
- la décision finale ;
- les raisons de revue ;
- les limites rencontrées.

Sans trace, il est presque impossible de déboguer un système multi-agent.

## 9. Boucle de coordination

Le flux général est le suivant :

```mermaid
flowchart TD
    A[Demande utilisateur] --> B[Analyse de tâche]
    B --> C[Sélection des agents]
    C --> D[Construction du plan]
    D --> E[Exécution des spécialistes]
    E --> F[Agrégation]
    F --> G{Conflit ou risque élevé ?}
    G -- Oui --> H[Reviewer]
    G -- Non --> I[Synthèse directe]
    H --> I
    I --> J[Réponse finale + trace]
```

## 10. Règles de conception

Un coordinateur professionnel doit suivre ces règles.

### Règle 1 — Déterminisme d'abord

La sélection des agents doit être aussi stable que possible.  
Pour une même tâche, le système doit produire un plan comparable.

### Règle 2 — Limites explicites

Le coordinateur doit avoir :

- un nombre maximum de rounds ;
- une liste d'agents autorisés ;
- une politique de revue ;
- une stratégie de fallback.

### Règle 3 — Contexte minimal

Chaque agent reçoit ce dont il a besoin, pas plus.

### Règle 4 — Conflits visibles

Un conflit ne doit pas être caché dans une réponse fluide.

### Règle 5 — Test sans LLM

La logique de coordination doit être testable sans dépendre d'un modèle externe.  
Le LLM peut être branché plus tard, mais le workflow doit être validé avant.

## 11. Exemple d'architecture

```mermaid
flowchart LR
    U[User Request] --> C[Coordinator]
    C --> R[Router]
    R --> P[Coordination Plan]
    P --> A1[Support Agent]
    P --> A2[Engineering Agent]
    P --> A3[Security Agent]
    A1 --> S[Synthesizer]
    A2 --> S
    A3 --> S
    S --> V{Needs Review?}
    V -- yes --> Rev[Reviewer Agent]
    V -- no --> F[Final Response]
    Rev --> F
    F --> T[Trace Store]
```

## 12. Application dans le lab

Le lab du jour implémente :

- un registre d'agents ;
- une tâche structurée ;
- un plan de coordination ;
- une sélection de spécialistes ;
- une exécution déterministe ;
- une détection de conflit ;
- un reviewer ;
- une synthèse finale ;
- une trace JSON.

Le but n'est pas de simuler un LLM.  
Le but est de construire l'ossature d'orchestration sur laquelle un LLM pourra ensuite être branché.

## 13. Lien avec MCP

La semaine 3 prépare l'introduction de MCP.

La coordination multi-agent doit rester indépendante du protocole d'outillage.  
Un agent peut utiliser :

- des outils Python locaux ;
- des APIs internes ;
- des outils exposés via MCP ;
- d'autres agents appelés comme outils.

Le coordinateur doit donc manipuler des contrats clairs plutôt que des détails d'implémentation.

## 14. Anti-patterns

### 14.1 Tous les agents répondent toujours

C'est coûteux et bruyant.  
Un bon coordinateur sélectionne.

### 14.2 Le reviewer décide sans critères

Une revue sans grille produit une opinion de plus.  
Il faut des critères : risque, conflit, incomplétude, action sensible.

### 14.3 Handoff non traçable

Si un agent transfère le contrôle sans trace, le système devient impossible à auditer.

### 14.4 Contexte global partagé

Tous les agents ne doivent pas voir toute la conversation.

### 14.5 Boucle infinie de désaccord

Un conflit doit avoir une limite d'itérations et un statut final clair.

## 15. Résumé

La coordination est une couche centrale de l'AI Engineering.

Elle permet de passer :

```text
plusieurs agents qui répondent
```

à :

```text
un système multi-agent contrôlé, observable et testable
```

La qualité d'un système multi-agent dépend souvent moins du nombre d'agents que de la qualité de sa coordination.

# Exercices — Coordination multi-agents

## Exercice 1 — Identifier les rôles

Pour chaque situation, indique quel rôle est principalement nécessaire :

1. Choisir entre un agent support et un agent billing.
2. Vérifier qu'une réponse finale ne promet pas une action interdite.
3. Transformer trois analyses locales en réponse utilisateur.
4. Décider qu'une tâche doit être traitée par deux agents en parallèle.
5. Transférer une conversation client vers un agent juridique.

Réponds avec l'un des rôles suivants :

- router ;
- manager ;
- specialist ;
- reviewer ;
- synthesizer ;
- handoff.

## Exercice 2 — Sélection d'agents

On dispose des agents suivants :

```text
support_agent: support client, réponse utilisateur
billing_agent: facturation, paiements, remboursements
engineering_agent: bugs, logs, API, incidents
security_agent: accès, permissions, données sensibles
reviewer_agent: cohérence, risque, conformité
```

Pour chaque demande, liste les agents utiles :

1. "Le client a été facturé deux fois et demande une réponse."
2. "Analyse cette erreur 500 sur l'API de paiement."
3. "Peux-tu expliquer pourquoi un utilisateur n'a plus accès à son compte ?"
4. "Prépare une réponse client sur un bug qui a exposé des données."
5. "Résume une note produit sans risque particulier."

## Exercice 3 — Contexte minimal

Voici une demande :

```text
Un client entreprise signale une facture de 1 200 € au lieu de 120 €.
Il est très mécontent. Il mentionne aussi qu'un développeur a vu une erreur API hier.
Il demande un remboursement immédiat et une explication technique.
```

Définis le contexte minimal à transmettre :

1. à `billing_agent` ;
2. à `engineering_agent` ;
3. à `support_agent` ;
4. à `reviewer_agent`.

## Exercice 4 — Détection de conflit

Deux agents produisent les observations suivantes :

```text
engineering_agent:
- status: resolved
- finding: "L'incident API est corrigé depuis 10h."

support_agent:
- status: unresolved
- finding: "Le client indique que le problème est encore présent à 11h."
```

1. Y a-t-il conflit ?
2. Quelle décision doit prendre le coordinateur ?
3. Quelle réponse finale ne doit surtout pas être produite ?

## Exercice 5 — Politique de revue

Propose une règle simple qui déclenche `reviewer_agent` dans les cas suivants :

- risque élevé ;
- désaccord entre agents ;
- action sensible ;
- manque d'information ;
- réponse externe à un client entreprise.

## Exercice 6 — Trace d'exécution

Écris une trace JSON minimale pour une coordination qui :

- reçoit une tâche `task-42` ;
- sélectionne `support_agent` et `billing_agent` ;
- détecte aucun conflit ;
- ne déclenche pas de revue ;
- produit une réponse finale.

## Exercice 7 — Amélioration du lab

Dans le fichier `labs/multi_agent_coordinator.py`, ajoute un nouvel agent `legal_agent`.

Il doit être sélectionné lorsqu'une tâche contient un domaine `legal`.

Ajoute un test qui vérifie que :

- `legal_agent` est sélectionné ;
- la trace contient son résultat ;
- une revue est déclenchée si le risque est `high`.

# Questions d'entretien — Coordination multi-agents

## Question 1

Pourquoi la coordination est-elle une responsabilité distincte des agents spécialistes ?

## Question 2

Quelle différence fais-tu entre router un message et effectuer un handoff ?

## Question 3

Quels sont les risques d'un système où tous les agents reçoivent tout le contexte ?

## Question 4

Comment détecterais-tu un conflit entre deux agents ?

## Question 5

Dans quel cas déclencherais-tu un reviewer ?

## Question 6

Pourquoi faut-il limiter le nombre de tours dans un système multi-agent ?

## Question 7

Comment testerais-tu une logique de coordination sans appeler un LLM ?

## Question 8

Quels éléments doivent apparaître dans une trace multi-agent ?

## Question 9

Pourquoi un système manager-worker peut-il être plus simple à auditer qu'un réseau de handoffs libres ?

## Question 10

Comment préparerais-tu une coordination multi-agent pour intégrer MCP plus tard ?

# Challenge — Coordinateur multi-agent contrôlé

## Contexte

Tu construis le moteur de coordination d'un assistant IA interne pour une équipe SaaS.

L'assistant peut déléguer à plusieurs agents :

- `support_agent` ;
- `billing_agent` ;
- `engineering_agent` ;
- `security_agent` ;
- `product_agent` ;
- `reviewer_agent`.

L'objectif est de produire une réponse finale fiable à partir d'une demande utilisateur.

## Mission

Améliore le lab pour supporter un workflow de coordination complet.

## Exigences fonctionnelles

Le coordinateur doit :

1. recevoir une tâche structurée ;
2. sélectionner les agents utiles ;
3. construire un plan d'exécution ;
4. exécuter les agents sélectionnés ;
5. collecter les observations ;
6. détecter les conflits ;
7. déclencher une revue si nécessaire ;
8. générer une réponse finale ;
9. produire une trace JSON.

## Règles de coordination

Le reviewer doit être déclenché si :

- le risque est `high` ;
- au moins un conflit est détecté ;
- la tâche contient le domaine `security` ;
- la réponse finale contient une action sensible ;
- aucun agent spécialiste n'a été sélectionné.

## Règles de contexte

Chaque agent ne doit recevoir que :

- l'identifiant de tâche ;
- l'objectif ;
- les domaines liés à sa compétence ;
- le niveau de risque ;
- les contraintes utiles.

## Règles de sortie

La sortie finale doit contenir :

```json
{
  "task_id": "string",
  "status": "completed | needs_review | needs_clarification",
  "selected_agents": ["string"],
  "review_performed": true,
  "conflicts": ["string"],
  "final_answer": "string",
  "trace": []
}
```

## Contraintes techniques

- Ne pas utiliser de dépendance externe.
- Ne pas appeler d'API.
- Garder le code testable avec `python test_multi_agent_coordinator.py`.
- Les décisions doivent être déterministes.
- Le code doit rester lisible.

## Bonus

Ajoute une politique de quorum :

- si trois agents ou plus sont sélectionnés ;
- et si deux agents convergent sur un statut identique ;
- le coordinateur peut produire une synthèse plus confiante ;
- sauf si `security_agent` ou `reviewer_agent` signale un blocage.

# Références — Coordination multi-agents

## Documentation principale

- OpenAI Agents SDK — Agent orchestration  
  https://openai.github.io/openai-agents-python/multi_agent/

- OpenAI Agents SDK — Handoffs  
  https://openai.github.io/openai-agents-python/handoffs/

- OpenAI Agents SDK — Agents  
  https://openai.github.io/openai-agents-python/agents/

- OpenAI — A practical guide to building agents  
  https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/

## Préparation MCP

- Model Context Protocol — Specification  
  https://modelcontextprotocol.io/specification/

- Model Context Protocol — Tools  
  https://modelcontextprotocol.io/specification/2025-06-18/server/tools

## Concepts à retenir

- Un handoff transfère le contrôle conversationnel.
- Un agent appelé comme outil exécute une sous-tâche sans nécessairement prendre le contrôle.
- Un manager central améliore souvent la traçabilité.
- Une coordination multi-agent doit être bornée, observable et testable.
- MCP est utile pour standardiser l'accès aux outils et contextes, mais il ne remplace pas la politique de coordination applicative.

## Lectures complémentaires

- Patterns : manager-worker, router-specialist, critic-reviewer, pipeline, debate.
- Concepts : traces, handoffs, context filtering, conflict resolution, deterministic orchestration.

# Corrigé — Exercices — Coordination multi-agents

## Exercice 1 — Identifier les rôles

1. Choisir entre support et billing : **router**.
2. Vérifier une promesse interdite : **reviewer**.
3. Transformer trois analyses locales : **synthesizer**.
4. Décider d'un traitement parallèle : **manager**.
5. Transférer vers juridique : **handoff**.

## Exercice 2 — Sélection d'agents

1. Facturation double + réponse client :
   - `billing_agent`
   - `support_agent`
   - éventuellement `reviewer_agent` si remboursement ou client entreprise.

2. Erreur 500 sur API de paiement :
   - `engineering_agent`
   - `billing_agent`
   - `reviewer_agent` si impact client ou risque élevé.

3. Utilisateur sans accès :
   - `security_agent`
   - `support_agent`
   - éventuellement `engineering_agent` si bug technique.

4. Bug exposant des données :
   - `engineering_agent`
   - `security_agent`
   - `support_agent`
   - `reviewer_agent`.

5. Résumé note produit sans risque :
   - `product_agent`
   - éventuellement `support_agent` si réponse utilisateur attendue.

## Exercice 3 — Contexte minimal

### billing_agent

```text
task_id
objectif: vérifier la facture
montant attendu: 120 €
montant facturé: 1 200 €
demande client: remboursement immédiat
contrainte: ne pas promettre de remboursement sans validation
risk_level: medium/high
```

### engineering_agent

```text
task_id
objectif: analyser l'erreur API mentionnée
indice: erreur API observée hier
lien potentiel: facturation incorrecte
contrainte: produire une hypothèse technique vérifiable
risk_level: medium
```

### support_agent

```text
task_id
objectif: préparer une réponse client
client: entreprise
sentiment: mécontent
faits confirmés: facture contestée, erreur API mentionnée
contrainte: empathie sans engagement non autorisé
risk_level: high
```

### reviewer_agent

```text
task_id
objectif: vérifier cohérence et risque
résultats billing/support/engineering
risques: remboursement, client entreprise, incident possible
contrainte: signaler promesses interdites et incertitudes
```

## Exercice 4 — Détection de conflit

1. Oui, il y a conflit : `resolved` vs `unresolved`.
2. Le coordinateur doit déclencher une revue ou demander une observation supplémentaire.
3. La réponse à éviter est : "Le problème est corrigé et tout fonctionne", car elle contredit le signal client.

## Exercice 5 — Politique de revue

Exemple de règle :

```python
requires_review = (
    risk_level == "high"
    or has_conflict
    or contains_sensitive_action
    or missing_required_information
    or external_enterprise_response
)
```

## Exercice 6 — Trace JSON minimale

```json
{
  "task_id": "task-42",
  "selected_agents": ["support_agent", "billing_agent"],
  "events": [
    {"type": "task_received", "task_id": "task-42"},
    {"type": "agents_selected", "agents": ["support_agent", "billing_agent"]},
    {"type": "agent_completed", "agent": "support_agent"},
    {"type": "agent_completed", "agent": "billing_agent"},
    {"type": "conflict_check", "conflicts": []},
    {"type": "final_answer_created"}
  ],
  "review_performed": false,
  "status": "completed"
}
```

## Exercice 7 — Amélioration du lab

Une solution possible :

```python
legal_agent = SpecialistAgent(
    name="legal_agent",
    domains=["legal"],
    default_status="caution",
    summary_template="Legal review required for task {task_id}."
)
```

Puis ajouter `legal_agent` au registre.

Test attendu :

```python
def test_legal_agent_is_selected_and_high_risk_triggers_review():
    registry = build_default_registry()
    registry.register(SpecialistAgent(
        name="legal_agent",
        domains=["legal"],
        default_status="caution",
        summary_template="Legal review required for task {task_id}."
    ))
    coordinator = MultiAgentCoordinator(registry)
    task = CoordinationTask(
        task_id="legal-1",
        objective="Review contractual wording",
        domains=["legal"],
        risk_level="high",
        constraints=[]
    )
    result = coordinator.run(task)
    assert "legal_agent" in result.selected_agents
    assert result.review_performed is True
    assert any(item.agent == "legal_agent" for item in result.observations)
```

# Corrigé — Questions d'entretien — Coordination multi-agents

## Question 1

La coordination est distincte parce qu'un agent spécialiste produit une réponse locale, alors que le coordinateur gère le système global : sélection des agents, ordre d'exécution, contexte partagé, conflits, limites, revue et trace.

## Question 2

Router consiste à choisir le bon agent ou les bons agents.  
Un handoff transfère le contrôle conversationnel à un autre agent.  
Le routing peut rester interne au manager, alors que le handoff change l'agent actif.

## Question 3

Partager tout le contexte augmente les coûts, introduit du bruit, peut exposer des informations inutiles, dégrade le focus et rend le debug plus difficile. Cela peut aussi créer des réponses contaminées par des informations qui ne concernent pas l'agent.

## Question 4

On peut détecter un conflit en comparant les statuts, les décisions, les contraintes ou les affirmations critiques. Dans un système simple, des statuts incompatibles comme `resolved` et `unresolved` suffisent à déclencher un conflit.

## Question 5

Un reviewer doit être déclenché en cas de risque élevé, désaccord, action sensible, manque d'information, domaine sécurité/juridique ou réponse externe engageante.

## Question 6

Les limites de tours évitent les boucles infinies, réduisent les coûts, améliorent la prévisibilité et forcent le système à produire un statut final clair.

## Question 7

On peut remplacer les agents LLM par des agents déterministes en Python. Chaque agent retourne une observation stable. Les tests vérifient la sélection, les conflits, la revue, la trace et le statut final.

## Question 8

Une trace doit contenir la tâche, les agents sélectionnés, les décisions de routing, les appels d'agents, les observations, les conflits, la revue éventuelle, la synthèse et le statut final.

## Question 9

Un manager-worker centralise la décision. Il est donc plus facile d'inspecter qui a été appelé, pourquoi, dans quel ordre et avec quel résultat. Un réseau libre de handoffs peut être plus flexible mais plus difficile à auditer.

## Question 10

Il faut isoler la politique de coordination des outils concrets. MCP peut ensuite fournir des tools, resources ou prompts, mais le coordinateur doit continuer à manipuler des contrats : tâche, plan, observation, conflit, résultat.

# Corrigé — Challenge — Coordinateur multi-agent contrôlé

## Approche recommandée

Une bonne solution consiste à séparer cinq responsabilités :

1. `AgentRegistry` : connaît les agents disponibles.
2. `CoordinationPolicy` : décide des règles de revue et de conflit.
3. `MultiAgentCoordinator` : orchestre le workflow.
4. `SpecialistAgent` : produit une observation locale.
5. `CoordinationResult` : expose une sortie structurée.

## Exemple de politique

```python
def needs_review(task, selected_agents, conflicts, sensitive_action):
    return (
        task.risk_level == "high"
        or bool(conflicts)
        or "security_agent" in selected_agents
        or sensitive_action
        or not selected_agents
    )
```

## Gestion du contexte minimal

Chaque agent reçoit une vue filtrée :

```python
{
    "task_id": task.task_id,
    "objective": task.objective,
    "domains": matching_domains,
    "risk_level": task.risk_level,
    "constraints": task.constraints,
}
```

## Sortie attendue

La sortie doit être sérialisable :

```python
result.to_dict()
```

Exemple :

```json
{
  "task_id": "task-001",
  "status": "needs_review",
  "selected_agents": ["support_agent", "security_agent"],
  "review_performed": true,
  "conflicts": [],
  "final_answer": "Review completed. Security constraints must be respected.",
  "trace": [
    {"type": "task_received", "task_id": "task-001"},
    {"type": "agents_selected", "agents": ["support_agent", "security_agent"]},
    {"type": "review_performed", "agent": "reviewer_agent"}
  ]
}
```

## Critères de validation

La solution est correcte si :

- les agents sont sélectionnés à partir des domaines ;
- le reviewer est déclenché selon les règles ;
- la trace explique les décisions ;
- les conflits ne sont pas masqués ;
- la sortie est déterministe ;
- les tests passent sans dépendance externe.

## Extension quorum

Une implémentation simple :

```python
statuses = [observation.status for observation in observations]
most_common_status = max(set(statuses), key=statuses.count)

if len(observations) >= 3 and statuses.count(most_common_status) >= 2:
    confidence = "medium"
else:
    confidence = "low"

if "blocked" in statuses:
    confidence = "low"
```

La règle de quorum ne doit jamais contourner un blocage sécurité ou une revue obligatoire.

# Notes formateur — Jour 2 — Coordination

## Intention pédagogique

Cette journée doit faire comprendre que le multi-agent est d'abord un problème d'architecture logicielle.  
Les apprenants doivent sortir de l'idée "plus d'agents = meilleur système".

Le message central est :

```text
La coordination est le produit.
```

## Points à insister

- Un agent spécialiste n'a pas la vision système.
- Le coordinateur doit rendre les décisions observables.
- Le routing est une décision.
- Le handoff est un transfert de contrôle.
- La revue est une politique, pas un agent magique.
- Le contexte doit être filtré.
- Les conflits doivent être visibles.

## Pièges fréquents

### Piège 1 — Appeler tous les agents

Les apprenants peuvent proposer d'appeler tous les agents à chaque demande.  
Corriger en parlant coût, bruit, latence et sécurité.

### Piège 2 — Faire confiance au reviewer sans critères

Un reviewer sans grille ne vaut pas mieux qu'une opinion supplémentaire.  
Demander toujours : "Qu'est-ce qui déclenche la revue ?"

### Piège 3 — Masquer les désaccords

Une belle synthèse qui cache un conflit est dangereuse.  
Le système doit exprimer l'incertitude.

### Piège 4 — Confondre state et trace

Le state représente l'état courant.  
La trace représente l'historique des décisions.

## Démonstration recommandée

Lancer :

```bash
python multi_agent_coordinator.py
python test_multi_agent_coordinator.py
```

Puis montrer :

- le plan ;
- les agents sélectionnés ;
- les observations ;
- la revue ;
- la trace JSON.

## Questions de discussion

- Quand faut-il préférer un handoff à un manager-worker ?
- Un reviewer doit-il être un LLM ou une règle déterministe ?
- Comment gérer une contradiction entre sécurité et produit ?
- Quelle trace serait nécessaire pour auditer une réponse client ?
- Comment brancher des outils MCP sans changer la politique de coordination ?

## Critère de maîtrise

Un apprenant maîtrise la journée s'il peut concevoir un workflow multi-agent avec :

- sélection contrôlée ;
- contexte minimal ;
- conflit explicite ;
- revue conditionnelle ;
- trace exploitable ;
- tests unitaires.

## Lab — Code principal

In [ ]:
"""
Semaine 3 — Jour 2 — Coordination multi-agents.

Ce module illustre une coordination multi-agent déterministe.
Il n'appelle aucun LLM et ne dépend d'aucune bibliothèque externe.

Objectif pédagogique :
- séparer agents, registre, politique et coordinateur ;
- produire un plan de coordination explicite ;
- détecter les conflits ;
- déclencher une revue ;
- générer une trace JSON exploitable.
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import Dict, Iterable, List, Optional, Sequence
import json


VALID_RISK_LEVELS = {"low", "medium", "high"}
CONFLICTING_STATUS_PAIRS = {
    frozenset(("resolved", "unresolved")),
    frozenset(("approved", "blocked")),
    frozenset(("safe", "unsafe")),
    frozenset(("ready", "missing_information")),
}


@dataclass(frozen=True)
class CoordinationTask:
    """Tâche structurée reçue par le coordinateur."""

    task_id: str
    objective: str
    domains: List[str]
    risk_level: str = "low"
    constraints: List[str] = field(default_factory=list)
    sensitive_action: bool = False

    def __post_init__(self) -> None:
        if not self.task_id.strip():
            raise ValueError("task_id must not be empty")
        if not self.objective.strip():
            raise ValueError("objective must not be empty")
        if self.risk_level not in VALID_RISK_LEVELS:
            raise ValueError(f"risk_level must be one of {sorted(VALID_RISK_LEVELS)}")


@dataclass(frozen=True)
class AgentContext:
    """Vue filtrée transmise à un agent."""

    task_id: str
    objective: str
    domains: List[str]
    risk_level: str
    constraints: List[str]


@dataclass(frozen=True)
class AgentObservation:
    """Observation produite par un agent spécialiste."""

    agent: str
    status: str
    finding: str
    confidence: float
    domains: List[str]

    def to_dict(self) -> Dict[str, object]:
        return asdict(self)


@dataclass(frozen=True)
class CoordinationPlan:
    """Plan d'exécution construit par le coordinateur."""

    task_id: str
    objective: str
    selected_agents: List[str]
    execution_mode: str
    requires_review: bool
    max_rounds: int
    review_reasons: List[str]

    def to_dict(self) -> Dict[str, object]:
        return asdict(self)


@dataclass(frozen=True)
class CoordinationResult:
    """Résultat final de coordination."""

    task_id: str
    status: str
    selected_agents: List[str]
    review_performed: bool
    conflicts: List[str]
    final_answer: str
    observations: List[AgentObservation]
    trace: List[Dict[str, object]]

    def to_dict(self) -> Dict[str, object]:
        return {
            "task_id": self.task_id,
            "status": self.status,
            "selected_agents": list(self.selected_agents),
            "review_performed": self.review_performed,
            "conflicts": list(self.conflicts),
            "final_answer": self.final_answer,
            "observations": [observation.to_dict() for observation in self.observations],
            "trace": list(self.trace),
        }

    def to_json(self) -> str:
        return json.dumps(self.to_dict(), ensure_ascii=False, indent=2)


class SpecialistAgent:
    """Agent spécialiste déterministe.

    Dans un système réel, cette classe pourrait appeler un LLM ou un outil MCP.
    Ici, elle retourne une observation stable pour tester la coordination.
    """

    def __init__(
        self,
        name: str,
        domains: Sequence[str],
        default_status: str,
        summary_template: str,
        confidence: float = 0.8,
    ) -> None:
        self.name = name
        self.domains = list(domains)
        self.default_status = default_status
        self.summary_template = summary_template
        self.confidence = confidence

    def supports_any(self, domains: Iterable[str]) -> bool:
        requested = set(domains)
        return any(domain in requested for domain in self.domains)

    def build_context(self, task: CoordinationTask) -> AgentContext:
        matching_domains = [domain for domain in task.domains if domain in self.domains]
        return AgentContext(
            task_id=task.task_id,
            objective=task.objective,
            domains=matching_domains,
            risk_level=task.risk_level,
            constraints=list(task.constraints),
        )

    def run(self, context: AgentContext) -> AgentObservation:
        finding = self.summary_template.format(
            task_id=context.task_id,
            objective=context.objective,
            domains=", ".join(context.domains) or "general",
            risk_level=context.risk_level,
        )
        return AgentObservation(
            agent=self.name,
            status=self.default_status,
            finding=finding,
            confidence=self.confidence,
            domains=list(context.domains),
        )


class ReviewerAgent(SpecialistAgent):
    """Agent de revue déterministe."""

    def __init__(self) -> None:
        super().__init__(
            name="reviewer_agent",
            domains=["review"],
            default_status="reviewed",
            summary_template="Review completed for task {task_id}.",
            confidence=0.9,
        )

    def review(
        self,
        task: CoordinationTask,
        observations: Sequence[AgentObservation],
        conflicts: Sequence[str],
        reasons: Sequence[str],
    ) -> AgentObservation:
        if conflicts:
            status = "needs_clarification"
            finding = (
                f"Review found unresolved coordination conflicts for {task.task_id}: "
                + "; ".join(conflicts)
            )
        elif task.risk_level == "high" or task.sensitive_action:
            status = "caution"
            finding = (
                f"Review completed for high-risk task {task.task_id}. "
                f"Reasons: {', '.join(reasons) or 'risk policy'}."
            )
        elif not observations:
            status = "missing_information"
            finding = f"Review could not validate {task.task_id} because no specialist was selected."
        else:
            status = "reviewed"
            finding = f"Review completed for task {task.task_id}; no blocking issue detected."

        return AgentObservation(
            agent=self.name,
            status=status,
            finding=finding,
            confidence=0.9,
            domains=["review"],
        )


class AgentRegistry:
    """Registre des agents disponibles."""

    def __init__(self) -> None:
        self._agents: Dict[str, SpecialistAgent] = {}

    def register(self, agent: SpecialistAgent) -> None:
        if agent.name in self._agents:
            raise ValueError(f"Agent already registered: {agent.name}")
        self._agents[agent.name] = agent

    def get(self, name: str) -> SpecialistAgent:
        return self._agents[name]

    def select_by_domains(self, domains: Sequence[str]) -> List[SpecialistAgent]:
        selected = [
            agent
            for agent in self._agents.values()
            if agent.name != "reviewer_agent" and agent.supports_any(domains)
        ]
        return sorted(selected, key=lambda agent: agent.name)

    @property
    def names(self) -> List[str]:
        return sorted(self._agents)


class CoordinationPolicy:
    """Politique de coordination.

    Elle décide :
    - du mode d'exécution ;
    - des raisons de revue ;
    - du statut final.
    """

    def __init__(self, max_rounds: int = 2) -> None:
        if max_rounds < 1:
            raise ValueError("max_rounds must be >= 1")
        self.max_rounds = max_rounds

    def execution_mode_for(self, selected_agents: Sequence[SpecialistAgent]) -> str:
        if len(selected_agents) <= 1:
            return "sequential"
        return "parallel"

    def review_reasons(
        self,
        task: CoordinationTask,
        selected_agent_names: Sequence[str],
        conflicts: Sequence[str],
    ) -> List[str]:
        reasons: List[str] = []

        if task.risk_level == "high":
            reasons.append("high_risk")

        if conflicts:
            reasons.append("conflict_detected")

        if task.sensitive_action:
            reasons.append("sensitive_action")

        if "security_agent" in selected_agent_names:
            reasons.append("security_domain")

        if not selected_agent_names:
            reasons.append("no_specialist_selected")

        if any("enterprise" in constraint.lower() for constraint in task.constraints):
            reasons.append("enterprise_external_response")

        return reasons

    def status_for(
        self,
        review_performed: bool,
        conflicts: Sequence[str],
        observations: Sequence[AgentObservation],
    ) -> str:
        if conflicts:
            return "needs_clarification"

        if not observations:
            return "needs_clarification"

        specialist_observations = [
            observation for observation in observations if observation.agent != "reviewer_agent"
        ]
        if not specialist_observations:
            return "needs_clarification"

        statuses = {observation.status for observation in observations}

        if "needs_clarification" in statuses or "missing_information" in statuses:
            return "needs_clarification"

        if "blocked" in statuses or "unsafe" in statuses:
            return "needs_review"

        if review_performed:
            return "needs_review"

        return "completed"


class MultiAgentCoordinator:
    """Coordinateur multi-agent minimal."""

    def __init__(self, registry: AgentRegistry, policy: Optional[CoordinationPolicy] = None) -> None:
        self.registry = registry
        self.policy = policy or CoordinationPolicy()

    def build_plan(self, task: CoordinationTask) -> CoordinationPlan:
        selected_agents = self.registry.select_by_domains(task.domains)
        selected_names = [agent.name for agent in selected_agents]
        preliminary_reasons = self.policy.review_reasons(task, selected_names, conflicts=[])

        return CoordinationPlan(
            task_id=task.task_id,
            objective=task.objective,
            selected_agents=selected_names,
            execution_mode=self.policy.execution_mode_for(selected_agents),
            requires_review=bool(preliminary_reasons),
            max_rounds=self.policy.max_rounds,
            review_reasons=preliminary_reasons,
        )

    def run(self, task: CoordinationTask) -> CoordinationResult:
        trace: List[Dict[str, object]] = [
            {"type": "task_received", "task_id": task.task_id, "domains": list(task.domains)}
        ]

        plan = self.build_plan(task)
        trace.append({"type": "plan_created", "plan": plan.to_dict()})

        observations: List[AgentObservation] = []
        for agent_name in plan.selected_agents:
            agent = self.registry.get(agent_name)
            context = agent.build_context(task)
            trace.append(
                {
                    "type": "agent_context_built",
                    "agent": agent.name,
                    "context": asdict(context),
                }
            )
            observation = agent.run(context)
            observations.append(observation)
            trace.append(
                {
                    "type": "agent_completed",
                    "agent": agent.name,
                    "status": observation.status,
                    "confidence": observation.confidence,
                }
            )

        conflicts = detect_conflicts(observations)
        trace.append({"type": "conflict_check", "conflicts": list(conflicts)})

        selected_names = [observation.agent for observation in observations]
        review_reasons = self.policy.review_reasons(task, selected_names, conflicts)
        review_performed = bool(review_reasons)

        if review_performed:
            reviewer = self.registry.get("reviewer_agent")
            if not isinstance(reviewer, ReviewerAgent):
                raise TypeError("reviewer_agent must be a ReviewerAgent")
            review_observation = reviewer.review(task, observations, conflicts, review_reasons)
            observations.append(review_observation)
            trace.append(
                {
                    "type": "review_performed",
                    "agent": reviewer.name,
                    "reasons": review_reasons,
                    "status": review_observation.status,
                }
            )
        else:
            trace.append({"type": "review_skipped", "reason": "policy_not_triggered"})

        status = self.policy.status_for(review_performed, conflicts, observations)
        final_answer = synthesize_final_answer(
            task=task,
            observations=observations,
            conflicts=conflicts,
            review_performed=review_performed,
        )

        trace.append({"type": "final_answer_created", "status": status})

        return CoordinationResult(
            task_id=task.task_id,
            status=status,
            selected_agents=selected_names,
            review_performed=review_performed,
            conflicts=list(conflicts),
            final_answer=final_answer,
            observations=observations,
            trace=trace,
        )


def detect_conflicts(observations: Sequence[AgentObservation]) -> List[str]:
    """Détecte des conflits simples à partir des statuts."""

    conflicts: List[str] = []
    statuses_by_agent = {observation.agent: observation.status for observation in observations}

    for left_agent, left_status in statuses_by_agent.items():
        for right_agent, right_status in statuses_by_agent.items():
            if left_agent >= right_agent:
                continue
            pair = frozenset((left_status, right_status))
            if pair in CONFLICTING_STATUS_PAIRS:
                conflicts.append(
                    f"{left_agent}:{left_status} conflicts with {right_agent}:{right_status}"
                )

    return conflicts


def synthesize_final_answer(
    task: CoordinationTask,
    observations: Sequence[AgentObservation],
    conflicts: Sequence[str],
    review_performed: bool,
) -> str:
    """Produit une réponse finale lisible et déterministe."""

    if not observations:
        return (
            f"Task {task.task_id} needs clarification because no specialist could be selected."
        )

    if conflicts:
        return (
            f"Task {task.task_id} needs clarification before a final decision. "
            f"Detected conflicts: {'; '.join(conflicts)}."
        )

    specialist_findings = [
        observation.finding
        for observation in observations
        if observation.agent != "reviewer_agent"
    ]

    review_note = ""
    if review_performed:
        review_observations = [
            observation.finding
            for observation in observations
            if observation.agent == "reviewer_agent"
        ]
        if review_observations:
            review_note = " Review note: " + review_observations[-1]

    joined_findings = " ".join(specialist_findings)
    return f"Task {task.task_id} coordinated successfully. {joined_findings}{review_note}".strip()


def build_default_registry() -> AgentRegistry:
    """Construit le registre utilisé dans les exercices et les tests."""

    registry = AgentRegistry()

    registry.register(
        SpecialistAgent(
            name="billing_agent",
            domains=["billing", "payment", "invoice"],
            default_status="ready",
            summary_template="Billing analysis for {task_id}: invoice/payment facts checked.",
            confidence=0.82,
        )
    )
    registry.register(
        SpecialistAgent(
            name="engineering_agent",
            domains=["engineering", "bug", "api", "incident"],
            default_status="resolved",
            summary_template="Engineering analysis for {task_id}: technical investigation completed.",
            confidence=0.78,
        )
    )
    registry.register(
        SpecialistAgent(
            name="product_agent",
            domains=["product", "roadmap", "feedback"],
            default_status="ready",
            summary_template="Product analysis for {task_id}: product implications summarized.",
            confidence=0.75,
        )
    )
    registry.register(
        SpecialistAgent(
            name="security_agent",
            domains=["security", "access", "permission", "data"],
            default_status="caution",
            summary_template="Security analysis for {task_id}: sensitive constraints identified.",
            confidence=0.86,
        )
    )
    registry.register(
        SpecialistAgent(
            name="support_agent",
            domains=["support", "customer", "response"],
            default_status="ready",
            summary_template="Support analysis for {task_id}: customer response prepared.",
            confidence=0.8,
        )
    )
    registry.register(ReviewerAgent())

    return registry


def demo() -> None:
    """Démonstration locale."""

    registry = build_default_registry()
    coordinator = MultiAgentCoordinator(registry)

    task = CoordinationTask(
        task_id="demo-001",
        objective="Prepare a customer response about a duplicated payment and possible API incident.",
        domains=["support", "billing", "engineering"],
        risk_level="medium",
        constraints=["External enterprise response", "Do not promise refund automatically."],
    )

    result = coordinator.run(task)
    print(result.to_json())


if __name__ == "__main__":
    demo()


## Lab — Tests

In [ ]:
"""
Semaine 3 — Jour 2 — Coordination multi-agents.

Ce module illustre une coordination multi-agent déterministe.
Il n'appelle aucun LLM et ne dépend d'aucune bibliothèque externe.

Objectif pédagogique :
- séparer agents, registre, politique et coordinateur ;
- produire un plan de coordination explicite ;
- détecter les conflits ;
- déclencher une revue ;
- générer une trace JSON exploitable.
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import Dict, Iterable, List, Optional, Sequence
import json


VALID_RISK_LEVELS = {"low", "medium", "high"}
CONFLICTING_STATUS_PAIRS = {
    frozenset(("resolved", "unresolved")),
    frozenset(("approved", "blocked")),
    frozenset(("safe", "unsafe")),
    frozenset(("ready", "missing_information")),
}


@dataclass(frozen=True)
class CoordinationTask:
    """Tâche structurée reçue par le coordinateur."""

    task_id: str
    objective: str
    domains: List[str]
    risk_level: str = "low"
    constraints: List[str] = field(default_factory=list)
    sensitive_action: bool = False

    def __post_init__(self) -> None:
        if not self.task_id.strip():
            raise ValueError("task_id must not be empty")
        if not self.objective.strip():
            raise ValueError("objective must not be empty")
        if self.risk_level not in VALID_RISK_LEVELS:
            raise ValueError(f"risk_level must be one of {sorted(VALID_RISK_LEVELS)}")


@dataclass(frozen=True)
class AgentContext:
    """Vue filtrée transmise à un agent."""

    task_id: str
    objective: str
    domains: List[str]
    risk_level: str
    constraints: List[str]


@dataclass(frozen=True)
class AgentObservation:
    """Observation produite par un agent spécialiste."""

    agent: str
    status: str
    finding: str
    confidence: float
    domains: List[str]

    def to_dict(self) -> Dict[str, object]:
        return asdict(self)


@dataclass(frozen=True)
class CoordinationPlan:
    """Plan d'exécution construit par le coordinateur."""

    task_id: str
    objective: str
    selected_agents: List[str]
    execution_mode: str
    requires_review: bool
    max_rounds: int
    review_reasons: List[str]

    def to_dict(self) -> Dict[str, object]:
        return asdict(self)


@dataclass(frozen=True)
class CoordinationResult:
    """Résultat final de coordination."""

    task_id: str
    status: str
    selected_agents: List[str]
    review_performed: bool
    conflicts: List[str]
    final_answer: str
    observations: List[AgentObservation]
    trace: List[Dict[str, object]]

    def to_dict(self) -> Dict[str, object]:
        return {
            "task_id": self.task_id,
            "status": self.status,
            "selected_agents": list(self.selected_agents),
            "review_performed": self.review_performed,
            "conflicts": list(self.conflicts),
            "final_answer": self.final_answer,
            "observations": [observation.to_dict() for observation in self.observations],
            "trace": list(self.trace),
        }

    def to_json(self) -> str:
        return json.dumps(self.to_dict(), ensure_ascii=False, indent=2)


class SpecialistAgent:
    """Agent spécialiste déterministe.

    Dans un système réel, cette classe pourrait appeler un LLM ou un outil MCP.
    Ici, elle retourne une observation stable pour tester la coordination.
    """

    def __init__(
        self,
        name: str,
        domains: Sequence[str],
        default_status: str,
        summary_template: str,
        confidence: float = 0.8,
    ) -> None:
        self.name = name
        self.domains = list(domains)
        self.default_status = default_status
        self.summary_template = summary_template
        self.confidence = confidence

    def supports_any(self, domains: Iterable[str]) -> bool:
        requested = set(domains)
        return any(domain in requested for domain in self.domains)

    def build_context(self, task: CoordinationTask) -> AgentContext:
        matching_domains = [domain for domain in task.domains if domain in self.domains]
        return AgentContext(
            task_id=task.task_id,
            objective=task.objective,
            domains=matching_domains,
            risk_level=task.risk_level,
            constraints=list(task.constraints),
        )

    def run(self, context: AgentContext) -> AgentObservation:
        finding = self.summary_template.format(
            task_id=context.task_id,
            objective=context.objective,
            domains=", ".join(context.domains) or "general",
            risk_level=context.risk_level,
        )
        return AgentObservation(
            agent=self.name,
            status=self.default_status,
            finding=finding,
            confidence=self.confidence,
            domains=list(context.domains),
        )


class ReviewerAgent(SpecialistAgent):
    """Agent de revue déterministe."""

    def __init__(self) -> None:
        super().__init__(
            name="reviewer_agent",
            domains=["review"],
            default_status="reviewed",
            summary_template="Review completed for task {task_id}.",
            confidence=0.9,
        )

    def review(
        self,
        task: CoordinationTask,
        observations: Sequence[AgentObservation],
        conflicts: Sequence[str],
        reasons: Sequence[str],
    ) -> AgentObservation:
        if conflicts:
            status = "needs_clarification"
            finding = (
                f"Review found unresolved coordination conflicts for {task.task_id}: "
                + "; ".join(conflicts)
            )
        elif task.risk_level == "high" or task.sensitive_action:
            status = "caution"
            finding = (
                f"Review completed for high-risk task {task.task_id}. "
                f"Reasons: {', '.join(reasons) or 'risk policy'}."
            )
        elif not observations:
            status = "missing_information"
            finding = f"Review could not validate {task.task_id} because no specialist was selected."
        else:
            status = "reviewed"
            finding = f"Review completed for task {task.task_id}; no blocking issue detected."

        return AgentObservation(
            agent=self.name,
            status=status,
            finding=finding,
            confidence=0.9,
            domains=["review"],
        )


class AgentRegistry:
    """Registre des agents disponibles."""

    def __init__(self) -> None:
        self._agents: Dict[str, SpecialistAgent] = {}

    def register(self, agent: SpecialistAgent) -> None:
        if agent.name in self._agents:
            raise ValueError(f"Agent already registered: {agent.name}")
        self._agents[agent.name] = agent

    def get(self, name: str) -> SpecialistAgent:
        return self._agents[name]

    def select_by_domains(self, domains: Sequence[str]) -> List[SpecialistAgent]:
        selected = [
            agent
            for agent in self._agents.values()
            if agent.name != "reviewer_agent" and agent.supports_any(domains)
        ]
        return sorted(selected, key=lambda agent: agent.name)

    @property
    def names(self) -> List[str]:
        return sorted(self._agents)


class CoordinationPolicy:
    """Politique de coordination.

    Elle décide :
    - du mode d'exécution ;
    - des raisons de revue ;
    - du statut final.
    """

    def __init__(self, max_rounds: int = 2) -> None:
        if max_rounds < 1:
            raise ValueError("max_rounds must be >= 1")
        self.max_rounds = max_rounds

    def execution_mode_for(self, selected_agents: Sequence[SpecialistAgent]) -> str:
        if len(selected_agents) <= 1:
            return "sequential"
        return "parallel"

    def review_reasons(
        self,
        task: CoordinationTask,
        selected_agent_names: Sequence[str],
        conflicts: Sequence[str],
    ) -> List[str]:
        reasons: List[str] = []

        if task.risk_level == "high":
            reasons.append("high_risk")

        if conflicts:
            reasons.append("conflict_detected")

        if task.sensitive_action:
            reasons.append("sensitive_action")

        if "security_agent" in selected_agent_names:
            reasons.append("security_domain")

        if not selected_agent_names:
            reasons.append("no_specialist_selected")

        if any("enterprise" in constraint.lower() for constraint in task.constraints):
            reasons.append("enterprise_external_response")

        return reasons

    def status_for(
        self,
        review_performed: bool,
        conflicts: Sequence[str],
        observations: Sequence[AgentObservation],
    ) -> str:
        if conflicts:
            return "needs_clarification"

        if not observations:
            return "needs_clarification"

        specialist_observations = [
            observation for observation in observations if observation.agent != "reviewer_agent"
        ]
        if not specialist_observations:
            return "needs_clarification"

        statuses = {observation.status for observation in observations}

        if "needs_clarification" in statuses or "missing_information" in statuses:
            return "needs_clarification"

        if "blocked" in statuses or "unsafe" in statuses:
            return "needs_review"

        if review_performed:
            return "needs_review"

        return "completed"


class MultiAgentCoordinator:
    """Coordinateur multi-agent minimal."""

    def __init__(self, registry: AgentRegistry, policy: Optional[CoordinationPolicy] = None) -> None:
        self.registry = registry
        self.policy = policy or CoordinationPolicy()

    def build_plan(self, task: CoordinationTask) -> CoordinationPlan:
        selected_agents = self.registry.select_by_domains(task.domains)
        selected_names = [agent.name for agent in selected_agents]
        preliminary_reasons = self.policy.review_reasons(task, selected_names, conflicts=[])

        return CoordinationPlan(
            task_id=task.task_id,
            objective=task.objective,
            selected_agents=selected_names,
            execution_mode=self.policy.execution_mode_for(selected_agents),
            requires_review=bool(preliminary_reasons),
            max_rounds=self.policy.max_rounds,
            review_reasons=preliminary_reasons,
        )

    def run(self, task: CoordinationTask) -> CoordinationResult:
        trace: List[Dict[str, object]] = [
            {"type": "task_received", "task_id": task.task_id, "domains": list(task.domains)}
        ]

        plan = self.build_plan(task)
        trace.append({"type": "plan_created", "plan": plan.to_dict()})

        observations: List[AgentObservation] = []
        for agent_name in plan.selected_agents:
            agent = self.registry.get(agent_name)
            context = agent.build_context(task)
            trace.append(
                {
                    "type": "agent_context_built",
                    "agent": agent.name,
                    "context": asdict(context),
                }
            )
            observation = agent.run(context)
            observations.append(observation)
            trace.append(
                {
                    "type": "agent_completed",
                    "agent": agent.name,
                    "status": observation.status,
                    "confidence": observation.confidence,
                }
            )

        conflicts = detect_conflicts(observations)
        trace.append({"type": "conflict_check", "conflicts": list(conflicts)})

        selected_names = [observation.agent for observation in observations]
        review_reasons = self.policy.review_reasons(task, selected_names, conflicts)
        review_performed = bool(review_reasons)

        if review_performed:
            reviewer = self.registry.get("reviewer_agent")
            if not isinstance(reviewer, ReviewerAgent):
                raise TypeError("reviewer_agent must be a ReviewerAgent")
            review_observation = reviewer.review(task, observations, conflicts, review_reasons)
            observations.append(review_observation)
            trace.append(
                {
                    "type": "review_performed",
                    "agent": reviewer.name,
                    "reasons": review_reasons,
                    "status": review_observation.status,
                }
            )
        else:
            trace.append({"type": "review_skipped", "reason": "policy_not_triggered"})

        status = self.policy.status_for(review_performed, conflicts, observations)
        final_answer = synthesize_final_answer(
            task=task,
            observations=observations,
            conflicts=conflicts,
            review_performed=review_performed,
        )

        trace.append({"type": "final_answer_created", "status": status})

        return CoordinationResult(
            task_id=task.task_id,
            status=status,
            selected_agents=selected_names,
            review_performed=review_performed,
            conflicts=list(conflicts),
            final_answer=final_answer,
            observations=observations,
            trace=trace,
        )


def detect_conflicts(observations: Sequence[AgentObservation]) -> List[str]:
    """Détecte des conflits simples à partir des statuts."""

    conflicts: List[str] = []
    statuses_by_agent = {observation.agent: observation.status for observation in observations}

    for left_agent, left_status in statuses_by_agent.items():
        for right_agent, right_status in statuses_by_agent.items():
            if left_agent >= right_agent:
                continue
            pair = frozenset((left_status, right_status))
            if pair in CONFLICTING_STATUS_PAIRS:
                conflicts.append(
                    f"{left_agent}:{left_status} conflicts with {right_agent}:{right_status}"
                )

    return conflicts


def synthesize_final_answer(
    task: CoordinationTask,
    observations: Sequence[AgentObservation],
    conflicts: Sequence[str],
    review_performed: bool,
) -> str:
    """Produit une réponse finale lisible et déterministe."""

    if not observations:
        return (
            f"Task {task.task_id} needs clarification because no specialist could be selected."
        )

    if conflicts:
        return (
            f"Task {task.task_id} needs clarification before a final decision. "
            f"Detected conflicts: {'; '.join(conflicts)}."
        )

    specialist_findings = [
        observation.finding
        for observation in observations
        if observation.agent != "reviewer_agent"
    ]

    review_note = ""
    if review_performed:
        review_observations = [
            observation.finding
            for observation in observations
            if observation.agent == "reviewer_agent"
        ]
        if review_observations:
            review_note = " Review note: " + review_observations[-1]

    joined_findings = " ".join(specialist_findings)
    return f"Task {task.task_id} coordinated successfully. {joined_findings}{review_note}".strip()


def build_default_registry() -> AgentRegistry:
    """Construit le registre utilisé dans les exercices et les tests."""

    registry = AgentRegistry()

    registry.register(
        SpecialistAgent(
            name="billing_agent",
            domains=["billing", "payment", "invoice"],
            default_status="ready",
            summary_template="Billing analysis for {task_id}: invoice/payment facts checked.",
            confidence=0.82,
        )
    )
    registry.register(
        SpecialistAgent(
            name="engineering_agent",
            domains=["engineering", "bug", "api", "incident"],
            default_status="resolved",
            summary_template="Engineering analysis for {task_id}: technical investigation completed.",
            confidence=0.78,
        )
    )
    registry.register(
        SpecialistAgent(
            name="product_agent",
            domains=["product", "roadmap", "feedback"],
            default_status="ready",
            summary_template="Product analysis for {task_id}: product implications summarized.",
            confidence=0.75,
        )
    )
    registry.register(
        SpecialistAgent(
            name="security_agent",
            domains=["security", "access", "permission", "data"],
            default_status="caution",
            summary_template="Security analysis for {task_id}: sensitive constraints identified.",
            confidence=0.86,
        )
    )
    registry.register(
        SpecialistAgent(
            name="support_agent",
            domains=["support", "customer", "response"],
            default_status="ready",
            summary_template="Support analysis for {task_id}: customer response prepared.",
            confidence=0.8,
        )
    )
    registry.register(ReviewerAgent())

    return registry


def demo() -> None:
    """Démonstration locale."""

    registry = build_default_registry()
    coordinator = MultiAgentCoordinator(registry)

    task = CoordinationTask(
        task_id="demo-001",
        objective="Prepare a customer response about a duplicated payment and possible API incident.",
        domains=["support", "billing", "engineering"],
        risk_level="medium",
        constraints=["External enterprise response", "Do not promise refund automatically."],
    )

    result = coordinator.run(task)
    print(result.to_json())


if __name__ == "__main__":
    demo()


## Exécution recommandée

Dans un dépôt local, placer les fichiers du lab dans le même dossier puis exécuter :

```bash
python multi_agent_coordinator.py
python test_multi_agent_coordinator.py
```